# 4.04 Arboles Azarosos — Validacion con multiples semillas del Top-N del Grid Search

Toma el ranking de `z422` (`gridsearch_local.txt`), se queda con las **mejores `PARAM$top_n` combinaciones** (por ganancia del ensemble completo) y las vuelve a correr con **`PARAM$qty_semillas` semillas nuevas**, cada una con su propia particion train/validacion Y su propio sampling de features. Esto sirve para detectar si el ranking de `z422` esta dominado por el azar de una unica particion ("fluke") antes de elegir que combinacion usar para el semillerio final que se sube a Kaggle.

#### Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [1]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Drive already mounted at /content/.drive; to attempt to forcibly remount, call drive.mount("/content/.drive", force_remount=True).


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

In [2]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"

---

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [1]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sat Aug 22 12:00:35 AM 2026"

In [2]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671290,35.9,1473300,78.7,1473300,78.7
Vcells,1242666,9.5,8388608,64.0,1978712,15.1


In [3]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia, cuantas semillas nuevas correr, y cuantas combinaciones del ranking de `z422` testear

In [4]:
PARAM <- list()
PARAM$semilla_primigenia <- 346321
PARAM$training_pct <- 70L # split train/validacion DENTRO de 202107, no toca Kaggle

PARAM$rpart$cp <- -1 # fijo, igual que en z422

PARAM$num_trees_max <- 8 # arboles por combinacion, igual que z422
PARAM$grabar <- c(1, 2, 4, 8) # puntos del ensemble donde se mide la ganancia, igual que z422

PARAM$top_n <- 10 # cuantas de las mejores combinaciones de z422 se re-testean

PARAM$qty_semillas <- 5 # cuantas semillas nuevas se corren por combinacion

# genero PARAM$qty_semillas semillas nuevas, reproducibles a partir de la primigenia
set.seed(PARAM$semilla_primigenia)
PARAM$semillas <- sample(1:1000000, PARAM$qty_semillas)

# donde esta el ranking de z422 del que se toma el Top-N
PARAM$archivo_grid_origen <- "/content/buckets/b1/exp/exp422/gridsearch_local.txt"

In [5]:
PARAM

$semilla_primigenia
[1] 346321

$training_pct
[1] 70

$rpart
$rpart$cp
[1] -1


$num_trees_max
[1] 8

$grabar
[1] 1 2 4 8

$top_n
[1] 10

$qty_semillas
[1] 5

$semillas
[1] 793779 243568 813017 804630  62182

$archivo_grid_origen
[1] "/content/buckets/b1/exp/exp422/gridsearch_local.txt"

In [6]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp423"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [7]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

# me quedo solo con los datos que tienen clase conocida, es decir 202107
#  (202109 tiene clase_ternaria vacia, no sirve para validar localmente)
dataset <- dataset[clase_ternaria != ""]

### Top-N combinaciones del ranking de `z422`

In [8]:
tb_grid_origen <- fread(PARAM$archivo_grid_origen)

# ranking por ganancia del ensemble completo (ultimo punto de grabar en z422)
tb_top <- tb_grid_origen[arbolito == max(PARAM$grabar)]
setorder(tb_top, -ganancia)
#tb_top <- tb_top[1:PARAM$top_n]
tb_top <- tb_top[11:30]
tb_top

combo_id,feature_fraction,cp,minsplit,minbucket,maxdepth,arbolito,ganancia
<int>,<dbl>,<int>,<int>,<int>,<int>,<int>,<dbl>
49,0.5,-1,200,100,6,8,487916667
69,0.7,-1,500,100,10,8,487500000
54,0.5,-1,200,150,10,8,487250000
14,0.3,-1,500,100,8,8,487166667
72,0.7,-1,500,150,10,8,487083333
77,0.7,-1,200,100,8,8,486750000
11,0.3,-1,500,50,8,8,486166667
29,0.5,-1,1000,50,8,8,485833333
53,0.5,-1,200,150,8,8,485666667


### Particion train / validacion local (70/30 estratificada por clase_ternaria)
A diferencia de `z422`, aca se arma una particion **nueva por cada semilla**, para medir la variabilidad real del proceso completo (no solo la del sampling de features).

In [9]:
# particionar agrega una columna llamada fold a un dataset
#  que consiste en una particion estratificada segun agrupa
particionar <- function(data, division, agrupa = "", campo = "fold", start = 1, seed = NA) {
  if (!is.na(seed)) set.seed(seed)

  bloque <- unlist(mapply(function(x, y) {
    rep(y, x)
  }, division, seq(from = start, length.out = length(division))))

  data[, (campo) := sample(rep(bloque, ceiling(.N / length(bloque))))[1:.N],
    by = agrupa
  ]
}

### Funcion que entrena el ensemble y devuelve la ganancia en validacion
Igual que en `z422`, salvo que ahora la semilla es un parametro (antes era siempre `PARAM$semilla_primigenia`), y `dtr`/`dval`/`campos_buenos` cambian en cada semilla del loop principal.

In [10]:
# entrena el ensemble de PARAM$num_trees_max arboles para una combinacion de
#  hiperparametros dada, y devuelve la ganancia normalizada sobre dval en cada
#  punto de PARAM$grabar (una fila por punto, columnas arbolito y ganancia)
ArbolesAzarososGanancia <- function(feature_fraction, rpart_control, semilla) {

  # misma semilla para todas las combinaciones DE ESTA semilla, asi la unica
  #  diferencia entre combinaciones es el hiperparametro, no el azar
  set.seed(semilla)

  tb_pred <- dval[, list(numero_de_cliente, clase_ternaria)]
  tb_pred[, prob_acumulada := 0]

  resultados <- data.table(arbolito = integer(), ganancia = numeric())

  for (arbolito in seq(PARAM$num_trees_max)) {
    qty_campos_a_utilizar <- as.integer(length(campos_buenos) * feature_fraction)
    campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
    campos_random <- paste(campos_random, collapse= " + ")
    formulita <- paste0("clase_ternaria ~ ", campos_random)

    modelo <- rpart(formulita, data= dtr, xval= 0, control= rpart_control)
    prediccion <- predict(modelo, dval, type= "prob")
    tb_pred[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]

    if (!(arbolito %in% PARAM$grabar)) next

    # umbral sobre la SUMA acumulada, equivalente a promedio > 1/40
    umbral_corte <- arbolito / 40
    tb_pred[, Predicted := prob_acumulada > umbral_corte]

    ganancia_test <- tb_pred[, sum(ifelse(Predicted,
        ifelse(clase_ternaria == "BAJA+2", 975000, -25000),
        0))]

    # escalo la ganancia como si fuera todo el dataset (misma logica que z422)
    ganancia_test_normalizada <- ganancia_test / ((100 - PARAM$training_pct) / 100)

    resultados <- rbindlist(list( resultados,
      data.table(arbolito= arbolito, ganancia= ganancia_test_normalizada)
    ))
  }

  resultados
}

### Validacion: Top-N combinaciones x semillas nuevas

In [11]:
# archivo donde se guarda el checkpoint (que combinaciones+semilla ya se corrieron)
archivo_grid <- "gridsearch_semillas_11_30.txt"

if (file.exists(archivo_grid)) {
  tb_grid <- fread(archivo_grid)
} else {
  tb_grid <- data.table(
    combo_id = integer(),
    feature_fraction = numeric(),
    cp = numeric(),
    minsplit = integer(),
    minbucket = integer(),
    maxdepth = integer(),
    semilla = integer(),
    arbolito = integer(),
    ganancia = numeric()
  )
}

for (v_semilla in PARAM$semillas) {

  # nueva particion train/validacion para esta semilla
  particionar(dataset,
    division= c(PARAM$training_pct, 100L - PARAM$training_pct),
    agrupa= "clase_ternaria",
    seed= v_semilla
  )

  dtr  <- dataset[fold == 1] # 70% training
  dval <- dataset[fold == 2] # 30% validacion local

  campos_buenos <- copy(setdiff(colnames(dtr), c("clase_ternaria", "fold")))

  for (i in seq_len(nrow(tb_top))) {

    v_combo_id         <- tb_top[i, combo_id]
    v_feature_fraction <- tb_top[i, feature_fraction]
    v_minsplit         <- tb_top[i, minsplit]
    v_minbucket        <- tb_top[i, minbucket]
    v_maxdepth         <- tb_top[i, maxdepth]

    # si esta combinacion (combo, semilla) ya tiene TODOS los puntos de grabar, la salteo
    puntos_hechos <- tb_grid[
      combo_id == v_combo_id & semilla == v_semilla,
      arbolito
    ]
    if (all(PARAM$grabar %in% puntos_hechos)) next

    cat("semilla=", v_semilla, " combo=", v_combo_id, "\n")
    flush.console()

    rpart_control <- list(
      cp= PARAM$rpart$cp,
      minsplit= v_minsplit,
      minbucket= v_minbucket,
      maxdepth= v_maxdepth
    )

    resultados <- ArbolesAzarososGanancia(v_feature_fraction, rpart_control, v_semilla)

    tb_grid <- rbindlist(list( tb_grid, data.table(
      combo_id= v_combo_id,
      feature_fraction= v_feature_fraction,
      cp= PARAM$rpart$cp,
      minsplit= v_minsplit,
      minbucket= v_minbucket,
      maxdepth= v_maxdepth,
      semilla= v_semilla,
      arbolito= resultados$arbolito,
      ganancia= resultados$ganancia
    )))

    # grabo el checkpoint despues de cada (combo, semilla), para poder retomar
    #  exactamente desde aca si se corta la conexion
    fwrite(tb_grid, file= archivo_grid, sep= "\t")
  }
}

semilla= 243568  combo= 54 
semilla= 243568  combo= 14 
semilla= 243568  combo= 72 
semilla= 243568  combo= 77 
semilla= 243568  combo= 11 
semilla= 243568  combo= 29 
semilla= 243568  combo= 53 
semilla= 243568  combo= 67 
semilla= 243568  combo= 60 
semilla= 243568  combo= 42 
semilla= 243568  combo= 59 
semilla= 243568  combo= 58 
semilla= 243568  combo= 35 
semilla= 243568  combo= 74 
semilla= 243568  combo= 33 
semilla= 243568  combo= 47 
semilla= 243568  combo= 40 
semilla= 243568  combo= 63 
semilla= 813017  combo= 49 
semilla= 813017  combo= 69 
semilla= 813017  combo= 54 


### Resultado: ranking robusto (promedio +- desvio sobre las semillas nuevas)
Cada combinacion tiene hasta `PARAM$qty_semillas` mediciones de ganancia del ensemble completo (una por semilla). `ganancia_sd` alta relativo a `ganancia_media` es señal de que esa combinacion es inestable entre particiones, no solo un numero mas alto o mas bajo.

In [12]:
tb_final <- tb_grid[arbolito == max(PARAM$grabar)]

resumen <- tb_final[, .(
    ganancia_media = mean(ganancia),
    ganancia_sd    = sd(ganancia),
    qty_semillas   = .N
  ), by= .(combo_id, feature_fraction, minsplit, minbucket, maxdepth)]

setorder(resumen, -ganancia_media)
resumen

combo_id,feature_fraction,minsplit,minbucket,maxdepth,ganancia_media,ganancia_sd,qty_semillas
<int>,<dbl>,<int>,<int>,<int>,<dbl>,<dbl>,<int>
68,0.7,500,100,8,570933333,21340770,5
38,0.5,500,50,8,566616667,22680189,5
81,0.7,200,150,10,566166667,21727718,5
80,0.7,200,150,8,564016667,13123796,5
30,0.5,1000,50,10,563450000,19973298,5
71,0.7,500,150,8,563083333,17263380,5
57,0.7,1000,50,10,557483333,26289481,5
66,0.7,500,50,10,556666667,23176107,5
15,0.3,500,100,10,549533333,14437677,5


In [ ]:
format(Sys.time(), "%a %b %d %X %Y")